# Assignment No. 16: K-Nearest Neighbours

**Objective:** Classify animal type using the K-Nearest Neighbours algorithm and evaluate its performance.

## 1. Import libraries and load the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)
from sklearn.decomposition import PCA

RANDOM_STATE = 42
sns.set_theme(style='whitegrid', palette='colorblind')
data = pd.read_csv('Zoo.csv')
data.head()

In [ ]:
print(f'Shape: {data.shape}')
display(data.info())
display(data.describe(include='all').T)
print('Missing values by column:')
display(data.isna().sum().to_frame('missing_values'))
print('Target distribution:')
display(data['type'].value_counts().sort_index().to_frame('count'))

## 2. Exploratory data analysis and visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=data, x='type', ax=axes[0])
axes[0].set_title('Animal types')
axes[0].set_xlabel('Animal type')
axes[0].set_ylabel('Number of animals')

feature_columns = data.drop(columns=['animal name', 'type']).columns
correlation = data[feature_columns].corr()
sns.heatmap(correlation, cmap='vlag', center=0, ax=axes[1])
axes[1].set_title('Feature correlation matrix')
plt.tight_layout()
plt.show()

The predictors are categorical/binary animal characteristics. The name is an identifier, not a predictive feature, so it is excluded. The class-count plot checks balance, while the heatmap shows relationships among characteristics.

## 3. Preprocessing: missing values and outliers

In [ ]:
# Confirm numeric predictors and handle missing values with the median.
model_data = data.drop(columns=['animal name']).copy()
missing_before = model_data.isna().sum().sum()
for column in model_data.columns.drop('type'):
    model_data[column] = model_data[column].fillna(model_data[column].median())

# IQR screening is reported for completeness. For binary 0/1 predictors,
# values outside the valid domain are the meaningful data-quality check.
outlier_counts = {}
for column in feature_columns:
    q1, q3 = model_data[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    outlier_counts[column] = int(((model_data[column] < q1 - 1.5 * iqr) |
                                  (model_data[column] > q3 + 1.5 * iqr)).sum())

print(f'Missing values before treatment: {missing_before}')
print(f'Missing values after treatment: {model_data.isna().sum().sum()}')
print(f'Rows with IQR outlier flags: {sum(outlier_counts.values())}')
print('All predictors are binary indicators; no continuous outlier transformation is applied.')
display(pd.Series(outlier_counts, name='IQR_flags').to_frame())

## 4. Train-test split and feature scaling

In [ ]:
X = model_data.drop(columns='type')
y = model_data['type']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training rows: {len(X_train)} ({len(X_train) / len(X):.0%})')
print(f'Testing rows: {len(X_test)} ({len(X_test) / len(X):.0%})')

## 5. Select K and distance metric

In [ ]:
# Euclidean distance is appropriate after standardization. Odd k values
# reduce ties; cross-validation selects the value based on training data.
k_values = list(range(1, 22, 2))
cv_folds = min(5, int(y_train.value_counts().min()))
cv_scores = []
for k in k_values:
    candidate = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    scores = cross_val_score(candidate, X_train_scaled, y_train, cv=cv_folds, scoring='accuracy')
    cv_scores.append(scores.mean())

best_k = k_values[int(np.argmax(cv_scores))]
print(f'Best K: {best_k}')
print(f'Best mean {cv_folds}-fold CV accuracy: {max(cv_scores):.4f}')

plt.figure(figsize=(9, 5))
plt.plot(k_values, cv_scores, marker='o')
plt.axvline(best_k, color='crimson', linestyle='--', label=f'Selected K = {best_k}')
plt.title('Cross-validation accuracy for K selection')
plt.xlabel('Number of neighbours (K)')
plt.ylabel('Mean validation accuracy')
plt.xticks(k_values)
plt.legend()
plt.show()

## 6. Train and evaluate the KNN classifier

In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k, metric='euclidean')
knn.fit(X_train_scaled, y_train)
y_pred = knn.predict(X_test_scaled)

metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision (weighted)': precision_score(y_test, y_pred, average='weighted', zero_division=0),
    'Recall (weighted)': recall_score(y_test, y_pred, average='weighted', zero_division=0),
    'F1-score (weighted)': f1_score(y_test, y_pred, average='weighted', zero_division=0),
}
display(pd.Series(metrics).round(4).to_frame('score'))
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
labels = sorted(y.unique())
matrix = confusion_matrix(y_test, y_pred, labels=labels)
plt.figure(figsize=(7, 6))
sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('KNN confusion matrix')
plt.xlabel('Predicted type')
plt.ylabel('Actual type')
plt.show()

## 7. Decision-boundary visualization

In [ ]:
# KNN uses all scaled features. PCA projects those features to two dimensions
# so its predicted regions can be plotted on a 2D chart.
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
label_order = sorted(y.unique())
label_to_code = {label: code for code, label in enumerate(label_order)}
y_train_codes = y_train.map(label_to_code)
boundary_knn = KNeighborsClassifier(n_neighbors=best_k, metric='euclidean')
boundary_knn.fit(X_train_pca, y_train_codes)

x_min, x_max = X_train_pca[:, 0].min() - 1, X_train_pca[:, 0].max() + 1
y_min, y_max = X_train_pca[:, 1].min() - 1, X_train_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
grid_pred = boundary_knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(10, 7))
plt.contourf(xx, yy, grid_pred, alpha=0.22, cmap='tab10', levels=np.arange(len(label_order) + 1) - 0.5)
sns.scatterplot(x=X_test_pca[:, 0], y=X_test_pca[:, 1], hue=y_test, style=y_test,
                palette='tab10', s=85, edgecolor='black')
plt.title('KNN decision regions in the first two PCA components')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.legend(title='Animal type', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Interview Questions

### 1. What are the key hyperparameters in KNN?
The main hyperparameters are `n_neighbors` (K), `metric` and its metric parameters, `weights` (`uniform` or `distance`), and the train/test preprocessing choices such as feature scaling. K controls the bias-variance trade-off: small K gives flexible boundaries, while large K gives smoother boundaries.

### 2. What distance metrics can be used in KNN?
Common choices include Euclidean, Manhattan, Minkowski, Chebyshev, cosine, Hamming, and Mahalanobis distance. This assignment uses Euclidean distance after standardizing the numeric indicator features. The most suitable metric depends on the feature type and meaning of similarity.

## Conclusion
The Zoo data was inspected visually, cleaned for missing values, split using an 80/20 stratified split, scaled, and classified with KNN. The selected K was determined by cross-validation, and the final model was evaluated with accuracy, weighted precision, weighted recall, weighted F1-score, a classification report, a confusion matrix, and a PCA-based decision-region plot.